# పాఠం 18 (అనుసరణ): ఒక *మానవుడు* చర్యను అనుమతించిందని సాక్ష్యాలు

ఈ పాఠం **ఏజెంట్** ఏమి చేశాడు మరియు **గేట్** ఏమి నిర్ణయించిందో నిరూపిస్తుంది. ఈ నోట్‌బుక్ పొరపాటు భాగాన్ని చేరుస్తుంది: ఒక **పేరు ఉన్న మానవుడు** **ఖచ్చితమైన** చర్యను ఆమోదించాడని సాక్ష్యం — పూర్తి ప్రమాణచేతన చర్యపై వేరే, మానవుని ద్వారా చేతిగ్రహీత సంతకం, ఆఫ్‌లైన్‌లో పరిశీలించబడింది.

ఇక్కడ రెండు ఆర్టిఫాక్ట్లు ఉపయోగించే **అదే ఎన్‌వలప్ ఆకారం పాఠం సాక్ష్యాలా** ఉంటుంది: ఒక ఫ్లాట్ పలోడ్, అందులో `type` ఫీల్డ్ వుంది, ఇది నేరుగా ప్రమాణచేతన JCS బైట్లు మీద Ed25519 చేత సంతకలుపబడింది, ఒక నిర్మిత `signature` వਸਤువుతో జతచేయబడింది (మరియు సంతకల బైట్ల నుంచి విడదీయబడింది). ఆమోద సాక్ష్యం ఒక కొత్త `type` (`human.approval.v1`), ఈ చర్య రకం పక్కన ఉంటుంది, కాబట్టి ఒకే `verify_chain` మీ ప్రధాన నోట్‌బుక్‌లో మీరు నిర్మించిన కోడ్ పాథ్‌తో రెండు రకాల ఆర్టిఫాక్ట్లను కవర్ చేస్తుంది. ఈ మానవ-ఆమోద సాక్ష్యం సాంకేతిక డ్రాఫ్ట్-farley-acta-signed-receipts నిర్వచించిన ఒక సాక్ష్యం రకంగా కాకుండా ఇక్కడ నిర్వచించబడిన విద్యార్థిని సంయోజనం.

ప్రధాన నోట్‌బుక్‌లోని డెమో వెరిఫైర్‌కు ఒక ఉద్దేశపూర్వక అభివృద్ధి: ఇక్కడి వెరిఫైర్ `signature.key_id`ని సాక్ష్యంలో ఉన్న ఒక పబ్లిక్ కీని నమ్మడం కాకుండా, **పిన్ చేయబడిన కీ రిజిస్ట్రి**పై పరిష్కరిస్తుంది. ఇది పాఠం సొంత చెక్‌లిస్ట్ సూచించే ఉత్పత్తి దృష్టి ("పబ్లిష్ చెయ్యండి ధృవీకరణ పబ్లిక్ కీ"), మరియు ఇది జాలీ నిర్మాణాన్ని నిరాకరణగా మార్చే దిశగా ఉంటుంది, కాదు మీ-తన-ఖాతాతో మినహాయింపు చేయడం.

ఈ నోట్‌బుక్ నేర్పించే నియమం: **ఒక సంతకముల ఆమోదం ఆనుమతి కాదు.** ఆమతి ఉంటుందని వుందంటే ఆమోద సాక్ష్యం మరియు చర్య సాక్ష్యం ఇంకా అదే ప్రమాణచేతన్ చర్యను అమలు సమయంలో బద్ధలుపుచేస్తే, ఒక విధాన సంస్కరణ, కీ, మరియు గడువు ఇంకా చేలగా ఉంటే, మరియు ఆమోదం ఇంకా ఉపయోగించబడని వుంటే మాత్రమే వుంది. ప్రతి విఫలం ఒక **భిన్న కారణంతో** నిరాకరిస్తుంది, కాబట్టి మీరు *ఆనుమతి జాతీయత అంతమైనది* మరియు *అమలైన చర్య మారింది* మధ్య తేడాను చెప్పగలరు.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## ఖచ్చితమైన చర్య

ఆమోదం యొక్క యూనిట్ **కేనానికల్ చర్య ఆబ్జెక్టు** — "రిఫండ్ ఆమోదించు" అనే అస్పష్ట లేబుల్ కాదు, కానీ ఖచ్చితమైన, పూర్తిగా నిర్దిష్టమైన చర్య. మొత్తం ఆబ్జెక్టును సంతకం చేస్తే (అదంతా డైజెస్ట్ తీసుకుంటే) మేము తర్వాత ఇన్స్టెన్స్ చూపించగలం, మనిషి ఈ చర్యని మాత్రమే ఆమోదించాడని.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ఒక రవాణా, రెండు అధికారాలు

ప్రతి రసీదు పాఠం యొక్క రవాణా పత్రం: ఒక ఫ్లాట్ పలోడ్ `type` ఫీల్డ్‌తో, అలాగే `signature` ఆబ్జెక్ట్ (`alg`, `sig`, `key_id`)తో ఉంటుంది, ఇది సంతకం చేసిన బైట్స్‌లో భాగం కాదు. `verify_envelope` అనేది రెండు రసీదు రకాల కోసం పంచుకుంటున్న నిర్మాణాత్మక + సంతకం తనిఖీ; ఇది `signature.key_id`ని ఏ **పిన్ చేసిన కీ రిజిస్ట్రీ**తో కలిపి పార్సు చేస్తుందో ఆ అధికారాలను వేరు చేస్తుంది:

- **ఆమోద రసీదు** (`human.approval.v1`) — పేరు చెప్పబడిన ఆమోదదారు, పూర్తి కలిసిన చర్య **మరియు దాని డైజెస్ట్**, `policy_version`, జారీ + గడువు తేది సారాలు ఉన్నాయి. ఒక్కసారి వినియోగం చైన్ స్థాయిలో ట్రాక్ చేయబడుతుంది.
- **చర్య రసీదు** (`agent.action.v1`) — ఏజెంట్ గుర్తింపు, `run_id`, అదే కలిసిన చర్య **డైజెస్ట్**, అమలు ఫలితం + తేది, మరియు `parent_approval_ref`: ఆమోదం యొక్క `receipt_hash`, పాఠం యొక్క చైన్‌లో ఉన్న `previous_receipt_hash`తో అదే నియమం.

పంచుకున్న `action_digest` ఫీల్డ్ అనేది బైండింగ్ ఆధారపడింది. `key_id` సంతకం ఆబ్జెక్టులో ఒక లుకప్ సూచన మాత్రమే: దాన్ని ఇతర పిన్ చేసిన కీలోకు తిరిగిపోయించడం సంతకం తనిఖీ విఫలమవుతుంది, కాబట్టి ఇది ఏమీ ఇవ్వదు.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: బైండింగ్ నిజంగా నిర్ణయించబడేది ఎక్కడ

`verify_chain` రెండు సంతకాల తనిఖీలపై సరళమైన చుట్టుపోటుగా **లేదు**. ఇది ఆ ఒక స్థలం, ఇక్కడ భాగస్వామ్య సరళమైన `action_digest`, ఆమోదంపై విధానము/కీ/కాలపరిమితి **తాజాదనం**, మరియు ఆమోదం యొక్క **ఒక్కసారిగా వినియోగం** జరుగుతాయి, ఇప్పుడు జరుగుతున్న చర్యకు వ్యతిరేకంగా ఒకటిగా తనిఖీ చేయబడుతాయి.

ప్రతి విఫలం ఒక **విభిన్న కారణంతో** నిరాకరిస్తుంది, కాబట్టి నిరాకరణ వాఙ్మయుడు తెలియజేయగలడు అధికారం పాతది అయినదో (విధానము మార్పిడి, కీ రొటేట్, ఆమోదం గడువు చివర, ఆమోదం వినియోగం) లేదా అమలైన చర్య ఇంకా చెలామణిలో ఉన్న ఆమోదం కింద మార్పు చెందినదో (డైజెస్ట్ ప్రత్యామ్నాయం).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## బైండింగ్ ఎలాంటి విషయాలను పట్టుకుంటుంది

దిగువ ప్రతి సందర్భం **వేరు కారణం**తో **బడితి**గా విఫలమవుతుంది. మొదటి బ్లాక్ క్లాసిక్ సెట్ (తప్పించటం, గందరగోళమైన డిప్యూటీ, రిప్లే, అధికారంపై మోసం, తప్పు ఆకారంలో ఇన్‌పుట్)ను సూచిస్తుంది. రెండో బ్లాక్ ఆ లక్షణాన్ని నిర్ధారించడానికి కాటు కట్టే జత:

- **పాత పైాధికారం** — సంతకం ఇంకా చెల్లుబాటు అవుతుంది, కానీ విధాన వెర్షన్ మారింది, ఆమోదం ఇచ్చే కీ నిర్దిష్ట రిజిస్ట్రీ నుండి తొలగించబడింది, లేదా ఆమోదం అమలు ముందు కాలయాపనైంది;
- **డైజెస్ట్ మార్పిడి** — చెల్లుబాటు సంతకం ఉన్న చర్య రశీదు, దీని `parent_approval_ref` ఒక *నిజమైన* ఆమోదాన్ని సూచిస్తుంది, కానీ ఆ ఆమోదం యొక్క కెనానికల్ చర్య డైజెస్ట్ అమలు అయ్యే చర్యకు సరిపోదు.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## ఇది ఏమి నిరూపిస్తుంది — మరియు 무엇 नहीं నిరూపిస్తుంది

**నిరూపిస్తుంది:** ఒక పేరున్న మనిషి ఈ *ఖచ్చితమైన కానోనికల్ చర్య* (పూర్తి చర్య + డైజెస్ట్, ఒక పిన్ చేసిన రిజిస్ట్రీ నుండి పరిష్కరించబడిన కీతో సంతకంతో)‌ను ఆమోదించారని, మరియు ఏజెంట్ అదే ఆమోదించబడిన చర్యను ఖచ్చితంగా అమలు చేశారని (అదే డైజెస్ట్, ఆమోదానికి బ్యాండ్ చేసిన రసీదు `receipt_hash` ద్వారా, పాఠం యొక్క సొంత చైన్ పరంపర ప్రకారం) — ఆమోదం యొక్క విధాన వెర్షన్, కీ, మరియు గడువు ఇంకా ప్రస్తుతం ఉన్నప్పుడు, ఖచ్చితంగా ఒకసారి. ఏవైనా రెండు వైపుల మార్పు ఉంటే, చైన్ మూసి పోతుంది, మరియు తిరస్కరణ కారణం మీకు **ఏ** లక్షణం విరిగిందో చెబుతుంది: పాత అధికార సంబంధిత లేదా మారిన చర్య.

**నిరూపించదను:** ఆమోద UI మనిషికి వారు సంతకం చేస్తున్నదని అనుకున్నదాన్ని చూపించిందని (WYSIWYS అనేది దీని సమస్య), కీ తిప్పే ముందు సుద్రుడిగాచేయబడలేదని లేదా దొంగిలించబడలేదని, లేదా దిగువ ప్రభావాలు చర్యకు సరిపోయాయని. సంతకం చేయబడినది = అనుమతించబడినది కాదు: పాత విధానం, మారిన కీ, గడువు గడిచిన గిట్టిన కాలం, లేదా వేరే డైజెస్ట్ పై సరైన సంతకం ఇక్కడ ఏమీ ఇవ్వదు.

ఈ రెండు రసీదు రకాలూ పాఠం యొక్క ఎన్ వలప్ మరియు ఒక `verify_chain` కోడ్ మార్గాన్ని ఉద్దేశపూర్వకంగా పంచుకుంటాయి: ప్రాథమిక నోట్బుక్లో చర్య రసీదు కోసం మీరు నిర్మించిన బంధం అదే కోడ్ ఇది మనిషి ఆమోదాన్ని తనిఖీ చేసే విధానం. ఒక నిర్ధారక ఒప్పందం, వేర్వేరు పిన్ చేసిన అధికారాలు, కానోనికల్ చర్య డైజెస్ట్ ద్వారా కలసిపోయినవి మరియు మరేదీ కాదు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
